# 03 — Gold Marts

Build three business-ready marts on top of silver. These are the tables BI dashboards and ML pipelines query.

| Mart | Grain | Use case |
|---|---|---|
| `customer_360` | one row per customer | unified view across all four verticals |
| `daily_transactions` | one row per (date, vertical, customer) | daily activity rollup, dashboard-ready |
| `loan_portfolio` | one row per loan | outstanding, missed-EMI count, default flag |

## Setup

In [ ]:
import sys
from pathlib import Path
if (Path.cwd() / "conf").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd() / "project" / "conf").exists():
    PROJECT_ROOT = Path.cwd() / "project"
else:
    raise RuntimeError("Run from inside project/ or repo root")
sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from conf.spark import get_spark, SILVER_DIR, GOLD_DIR

spark = get_spark("GoldMarts")
GOLD_DIR.mkdir(parents=True, exist_ok=True)

def silver(t): return spark.read.format("delta").load(str(SILVER_DIR / t))
def write_gold(df, t):
    target = GOLD_DIR / t
    (df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").save(str(target)))
    print(f"  gold.{t:24s} {spark.read.format('delta').load(str(target)).count():>5} rows")

## customer_360

Aggregate per-customer counts and totals from each vertical, then left-join onto the customer dimension. Uses **left joins** so customers with no activity in a given vertical still appear (with zeros).

In [ ]:
customers = silver("customers")

card_agg = (silver("card_transactions")
    .filter(F.col("status") == "approved")
    .groupBy("customer_id")
    .agg(F.count("*").alias("card_txn_count"),
         F.coalesce(F.sum("amount"), F.lit(0)).alias("card_spend_total"),
         F.sum(F.when(F.col("is_flagged"), 1).otherwise(0)).alias("card_flagged_count")))

loan_agg = (silver("loan_accounts")
    .groupBy("customer_id")
    .agg(F.count("*").alias("loan_count"),
         F.coalesce(F.sum("principal_amount"), F.lit(0)).alias("loan_principal_total")))

acct_agg = (silver("bank_accounts")
    .filter(F.col("status") == "active")
    .groupBy("customer_id")
    .agg(F.count("*").alias("account_count"),
         F.coalesce(F.sum("balance"), F.lit(0)).alias("account_balance_total")))

pay_agg = (silver("payments")
    .filter(F.col("status") == "success")
    .groupBy(F.col("sender_customer_id").alias("customer_id"))
    .agg(F.count("*").alias("payment_count"),
         F.coalesce(F.sum("amount"), F.lit(0)).alias("payment_total")))

c360 = (customers
    .join(card_agg, "customer_id", "left")
    .join(loan_agg, "customer_id", "left")
    .join(acct_agg, "customer_id", "left")
    .join(pay_agg,  "customer_id", "left")
    .na.fill(0, ["card_txn_count", "card_spend_total", "card_flagged_count",
                  "loan_count", "loan_principal_total",
                  "account_count", "account_balance_total",
                  "payment_count", "payment_total"]))

write_gold(c360, "customer_360")
c360.select("customer_id", "full_name", "city",
             "card_txn_count", "loan_count", "account_count", "payment_count").show(5, truncate=False)

## daily_transactions

Union activity from cards, accounts, and payments into one long table at the (date, customer, vertical) grain. This is what powers the daily dashboards.

In [ ]:
card_d = (silver("card_transactions")
    .filter(F.col("status") == "approved")
    .select(F.to_date("transaction_at").alias("txn_date"),
            F.col("customer_id"),
            F.lit("cards").alias("vertical"),
            F.col("amount")))

acct_d = (silver("account_transactions")
    .select(F.to_date("txn_at").alias("txn_date"),
            F.col("customer_id"),
            F.lit("accounts").alias("vertical"),
            F.col("amount")))

pay_d = (silver("payments")
    .filter(F.col("status") == "success")
    .select(F.to_date("initiated_at").alias("txn_date"),
            F.col("sender_customer_id").alias("customer_id"),
            F.lit("payments").alias("vertical"),
            F.col("amount")))

daily = (card_d.unionByName(acct_d).unionByName(pay_d)
    .groupBy("txn_date", "customer_id", "vertical")
    .agg(F.count("*").alias("txn_count"),
         F.sum("amount").alias("amount_total")))

write_gold(daily, "daily_transactions")
daily.orderBy(F.col("txn_date").desc()).show(8, truncate=False)

## loan_portfolio

One row per loan with current outstanding (principal − total paid) and a `is_default` flag using a simple rule: ≥3 missed EMIs **or** raw status of `defaulted`.

In [ ]:
loans = silver("loan_accounts")
rep   = silver("loan_repayments")

rep_agg = (rep.groupBy("loan_id")
    .agg(F.coalesce(F.sum("paid_amount"), F.lit(0)).alias("total_paid"),
         F.sum(F.when(F.col("status") == "missed", 1).otherwise(0)).alias("missed_emis"),
         F.count("*").alias("emis_billed")))

portfolio = (loans.join(rep_agg, "loan_id", "left")
    .na.fill(0, ["total_paid", "missed_emis", "emis_billed"])
    .withColumn("outstanding", F.col("principal_amount") - F.col("total_paid"))
    .withColumn("is_default",
                (F.col("missed_emis") >= 3) | (F.col("status") == "defaulted")))

write_gold(portfolio, "loan_portfolio")
portfolio.select("loan_id", "customer_id", "loan_type", "principal_amount",
                  "total_paid", "outstanding", "missed_emis", "is_default") \
         .orderBy(F.col("missed_emis").desc()) \
         .show(8, truncate=False)

## Sanity check

In [ ]:
for t in ["customer_360", "daily_transactions", "loan_portfolio"]:
    df = spark.read.format("delta").load(str(GOLD_DIR / t))
    print(f"{t:22s} {df.count():>5} rows, {len(df.columns)} cols")

## Summary

Three gold marts written. Continue with **04-sql-analytics.ipynb** to query them with Spark SQL.